# PoliMillionaire Hybrid RAG Pipeline — Qwen3.5-9B Q6_K_L GGUF

This notebook runs the same pipeline as the previous hybrid RAG notebook, but replaces the `transformers` LLM with a GGUF model loaded via `llama-cpp-python`:

```text
LLM: bartowski/Qwen_Qwen3.5-9B-GGUF
File: Qwen3.5-9B-Q6_K_L.gguf
Backend: llama.cpp / llama-cpp-python
```

Pipeline:

```text
Question + competition_name
↓
if competition_name == "Maths":
    deterministic SymPy tools
    ↓
    LLM tool router
    ↓
    if solved: return option_id
↓
SimpleWiki BM25 + SimpleWiki dense
KELM BM25 + KELM dense
↓
RRF fusion
↓
Cross-encoder reranker on CPU
↓
Qwen3.5-9B GGUF final answer selector
↓
option_id
```

Use this notebook in a **fresh Colab runtime**. Set your Hugging Face token in Colab Secrets as `HF_TOKEN` before running the model download cell.


## 1. Install dependencies


In [ ]:
# deps minime
!pip install -q huggingface_hub hnswlib bm25s

# forza wheel CUDA precompilato (NO compilazione)
!pip install -q --force-reinstall \
  llama-cpp-python \
  --index-url https://pypi.org/simple \
  --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu124

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.5/74.5 kB 2.6 MB/s eta 0:00:00


### Optional fallback: rebuild llama-cpp-python with CUDA


In [ ]:
# Run this ONLY if the wheel above fails or does not use the GPU.
# It can take several minutes.
# !CMAKE_ARGS="-DGGML_CUDA=on" FORCE_CMAKE=1 pip install --no-cache-dir --force-reinstall llama-cpp-python


## 2. Mount Drive, paths, and token setup


In [ ]:
from pathlib import Path
import os, sys, json, time, math, re, shutil, gc

try:
    from google.colab import drive, userdata
    drive.mount('/content/drive')
    IN_COLAB = True
except Exception:
    IN_COLAB = False
    userdata = None

PROJECT_ROOT = Path('/content/drive/MyDrive/nlp26') if IN_COLAB else Path.cwd()
SRC_DIR = PROJECT_ROOT / 'src'
API_CLIENT_DIR = PROJECT_ROOT / 'api_client'
DRIVE_INDEX_DIR = PROJECT_ROOT / 'indexes'
LOG_DIR = PROJECT_ROOT / 'logs'
LOG_DIR.mkdir(parents=True, exist_ok=True)

LOCAL_ROOT = Path('/content/nlp26_runtime') if IN_COLAB else PROJECT_ROOT / '.runtime'
LOCAL_INDEX_DIR = LOCAL_ROOT / 'indexes'
LOCAL_MODEL_DIR = Path('/content/models') if IN_COLAB else PROJECT_ROOT / 'models'
LOCAL_HF_CACHE = Path('/content/hf_cache') if IN_COLAB else PROJECT_ROOT / '.hf_cache'

for p in [SRC_DIR, API_CLIENT_DIR, PROJECT_ROOT]:
    if str(p) not in sys.path:
        sys.path.append(str(p))

# Hugging Face token: prefer Colab Secret HF_TOKEN, otherwise env var.
if IN_COLAB:
    try:
        token = userdata.get('HF_TOKEN')
        if token:
            os.environ['HF_TOKEN'] = token
    except Exception as e:
        print('Could not read Colab secret HF_TOKEN:', repr(e))

os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'
os.environ['HF_HOME'] = str(LOCAL_HF_CACHE)

print('PROJECT_ROOT:', PROJECT_ROOT, PROJECT_ROOT.exists())
print('SRC_DIR:', SRC_DIR, SRC_DIR.exists())
print('API_CLIENT_DIR:', API_CLIENT_DIR, API_CLIENT_DIR.exists())
print('DRIVE_INDEX_DIR:', DRIVE_INDEX_DIR, DRIVE_INDEX_DIR.exists())
print('LOCAL_INDEX_DIR:', LOCAL_INDEX_DIR)
print('HF_TOKEN set:', bool(os.environ.get('HF_TOKEN')))


In [ ]:
from google.colab import drive
#drive.flush_and_unmount()


## 3. Memory helpers


In [ ]:
import psutil

try:
    import torch
except Exception:
    torch = None

def mem_report(label=''):
    print(f"\n[MEM] {label}")
    vm = psutil.virtual_memory()
    print(f"CPU RAM: {vm.used/1024**3:.2f} / {vm.total/1024**3:.2f} GiB ({vm.percent:.1f}%)")
    if torch is not None and torch.cuda.is_available():
        print(f"GPU torch allocated: {torch.cuda.memory_allocated()/1024**3:.2f} GiB")
        print(f"GPU torch reserved:  {torch.cuda.memory_reserved()/1024**3:.2f} GiB")

def cleanup_memory():
    gc.collect()
    if torch is not None and torch.cuda.is_available():
        torch.cuda.empty_cache()
        try:
            torch.cuda.ipc_collect()
        except Exception:
            pass

mem_report('initial')


## 4. Copy index files from Drive to local Colab disk


In [ ]:
INDEX_FILES = {
    'simplewiki_bm25': 'simplewiki_160w_title2_stop_bm25.joblib',
    'simplewiki_dense_index': 'simplewiki_160w_dense_hnsw.index',
    'simplewiki_dense_meta': 'simplewiki_160w_dense_meta.joblib',
    'kelm_bm25': 'kelm_500k_stop_bm25.joblib',
    'kelm_dense_index': 'kelm_500k_dense_hnsw.index',
    'kelm_dense_meta': 'kelm_500k_dense_meta.joblib',
}

def copy_indexes_to_local():
    LOCAL_INDEX_DIR.mkdir(parents=True, exist_ok=True)
    out = {}
    for key, filename in INDEX_FILES.items():
        src = DRIVE_INDEX_DIR / filename
        dst = LOCAL_INDEX_DIR / filename
        if not src.exists():
            raise FileNotFoundError(f'Missing index file on Drive: {src}')
        if not dst.exists() or dst.stat().st_size != src.stat().st_size:
            print(f'Copying {filename} -> {dst}')
            shutil.copy2(src, dst)
        out[key] = dst
    return out

LOCAL_INDEX_FILES = copy_indexes_to_local()
for key, path in LOCAL_INDEX_FILES.items():
    print(f'{key:28s}', path.exists(), f'{path.stat().st_size/1024**2:.1f} MB', path)

mem_report('after local index cache')


## 5. Download and load Qwen3.5-9B Q6_K_L GGUF


In [ ]:
from huggingface_hub import hf_hub_download
from llama_cpp import Llama

# Recommended quality/memory compromise from the Qwen3.5-9B GGUF comparison.
MODEL_REPO = 'bartowski/Qwen_Qwen3.5-9B-GGUF'
MODEL_FILE = 'Qwen_Qwen3.5-9B-Q6_K_L.gguf'

# If you want the smaller Unsloth Q6_K instead, switch to:
# MODEL_REPO = 'unsloth/Qwen3.5-9B-GGUF'
# MODEL_FILE = 'Qwen3.5-9B-Q6_K.gguf'

MODEL_PATH = hf_hub_download(
    repo_id=MODEL_REPO,
    filename=MODEL_FILE,
    local_dir=str(LOCAL_MODEL_DIR),
    token=os.environ.get('HF_TOKEN'),
)

print('MODEL_PATH:', MODEL_PATH)
print('Model file size:', Path(MODEL_PATH).stat().st_size / 1024**3, 'GiB')
mem_report('after GGUF download')


In [ ]:
# Start conservative on a T4. Increase n_gpu_layers only after checking nvidia-smi.
N_CTX = 4096
N_GPU_LAYERS = -1      # try 35, then -1 if memory is stable
N_BATCH = 256
N_THREADS = 2

qwen35_llm = Llama(
    model_path=MODEL_PATH,
    n_ctx=N_CTX,
    n_gpu_layers=N_GPU_LAYERS,
    n_batch=N_BATCH,
    n_threads=N_THREADS,
    logits_all=False,
    verbose=False,
)

mem_report('after Qwen3.5 GGUF load')
!nvidia-smi


## 6. GGUF LLM wrapper


In [ ]:
def run_local_llm(prompt: str, max_new_tokens: int = 8, stop=None) -> str:
    if stop is None:
        stop = ['<|im_end|>', '<|endoftext|>']
    out = qwen35_llm(
        prompt,
        max_tokens=max_new_tokens,
        temperature=0.0,
        top_p=1.0,
        repeat_penalty=1.05,
        stop=stop,
    )
    return out['choices'][0]['text'].strip()

# Smoke test
prompt = """You are answering a multiple-choice question.
Return ONLY the numeric option id.

Question:
Who was the first president of the United States?

Options:
1. Abraham Lincoln
2. George Washington
3. Thomas Jefferson
4. John Adams

Answer:"""
print(run_local_llm(prompt, max_new_tokens=4))
mem_report('after LLM smoke test')


## 7. Load retrieval stack: embedding model, BM25, HNSW dense, reranker


In [ ]:
import numpy as np
import pandas as pd
import joblib
import hnswlib
from collections import defaultdict
from sentence_transformers import SentenceTransformer, CrossEncoder

EMBEDDING_MODEL_NAME = 'sentence-transformers/multi-qa-MiniLM-L6-cos-v1'
RERANKER_MODEL_NAME = 'cross-encoder/ms-marco-MiniLM-L-6-v2'

TOP_K_BM25 = 60
TOP_K_DENSE = 40
RRF_K = 60
RRF_TOP_K = 30
RERANK_TOP_K = 12
LLM_CONTEXT_K = 4
DOC_MAX_CHARS = 500
MAX_NEW_TOKENS_FINAL = 4
MAX_NEW_TOKENS_ROUTER = 80
MATH_COMPETITION_NAME = 'Maths'
PROMPT_VERSION = 'qwen35_9b_q6kl_gguf_v1'

embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME, device='cpu')
mem_report('after embedding model')


In [ ]:
def normalize_text(x):
    if x is None:
        return ''
    if isinstance(x, str):
        return x
    try:
        return json.dumps(x, ensure_ascii=False)
    except Exception:
        return str(x)

def simple_tokenize(text):
    return re.findall(r"[A-Za-z0-9_]+", normalize_text(text).lower())

def extract_doc_text(doc):
    if isinstance(doc, str):
        return doc
    if isinstance(doc, dict):
        for key in ['text', 'contents', 'content', 'passage', 'document', 'body', 'chunk']:
            if key in doc and doc[key]:
                return normalize_text(doc[key])
        return normalize_text(doc)
    return normalize_text(doc)

def extract_docs_from_loaded(obj):
    if isinstance(obj, dict):
        for key in ['docs', 'documents', 'corpus', 'texts', 'chunks', 'passages']:
            if key in obj and obj[key] is not None:
                return list(obj[key])
        for key in ['metadata', 'metas', 'meta']:
            if key in obj and isinstance(obj[key], (list, tuple)):
                return list(obj[key])
    if isinstance(obj, (list, tuple)):
        return list(obj)
    return None

def make_doc_id(source, idx):
    return f'{source}:{int(idx)}'

def make_result_item(source, idx, text, score=None, rank=None, method=None):
    return {
        'doc_id': make_doc_id(source, idx),
        'source': source,
        'idx': int(idx),
        'text': extract_doc_text(text),
        'score': float(score) if score is not None else None,
        'rank': int(rank) if rank is not None else None,
        'method': method,
    }

class SparseIndexAdapter:
    def __init__(self, path, source):
        self.path = Path(path)
        self.source = source
        self.obj = joblib.load(self.path)
        self.docs = extract_docs_from_loaded(self.obj)
        self.bm25 = None
        self.vectorizer = None
        self.matrix = None
        if isinstance(self.obj, dict):
            self.bm25 = self.obj.get('bm25') or self.obj.get('index') or self.obj.get('bm25_index')
            self.vectorizer = self.obj.get('vectorizer')
            self.matrix = self.obj.get('matrix') or self.obj.get('X') or self.obj.get('tfidf_matrix')
        else:
            self.bm25 = self.obj
        if self.docs is None:
            raise ValueError(f'Could not extract docs from {path}')
        print(f'[SparseIndexAdapter] {source}: docs={len(self.docs)} bm25={self.bm25 is not None} vectorizer={self.vectorizer is not None}')

    def search(self, query, top_k=50):
        tokens = simple_tokenize(query)
        # bm25s style or custom bm25 object
        if self.bm25 is not None:
            # Try bm25s retrieve API variants.
            for call in [
                lambda: self.bm25.retrieve([tokens], k=top_k),
                lambda: self.bm25.retrieve(tokens, k=top_k),
                lambda: self.bm25.get_top_n(tokens, self.docs, n=top_k),
            ]:
                try:
                    res = call()
                    # bm25s often returns (results, scores) arrays.
                    if isinstance(res, tuple) and len(res) == 2:
                        indices, scores = res
                        indices = np.array(indices).reshape(-1)[:top_k]
                        scores = np.array(scores).reshape(-1)[:top_k]
                        return [make_result_item(self.source, int(i), self.docs[int(i)], score=s, rank=r, method='bm25')
                                for r, (i, s) in enumerate(zip(indices, scores), start=1)]
                    # If returns docs directly, map by identity is impossible; return text-only pseudo indices.
                    if isinstance(res, list) and res and not isinstance(res[0], (int, np.integer)):
                        return [make_result_item(self.source, i, d, score=None, rank=i+1, method='bm25')
                                for i, d in enumerate(res[:top_k])]
                except Exception:
                    pass
            # rank_bm25/get_scores style
            try:
                scores = np.asarray(self.bm25.get_scores(tokens))
                idx = np.argsort(-scores)[:top_k]
                return [make_result_item(self.source, int(i), self.docs[int(i)], score=scores[int(i)], rank=r, method='bm25')
                        for r, i in enumerate(idx, start=1)]
            except Exception as e:
                raise RuntimeError(f'BM25 search failed for {self.source}: {e}')
        # sklearn TF-IDF fallback
        if self.vectorizer is not None and self.matrix is not None:
            qv = self.vectorizer.transform([query])
            scores = (self.matrix @ qv.T).toarray().reshape(-1)
            idx = np.argsort(-scores)[:top_k]
            return [make_result_item(self.source, int(i), self.docs[int(i)], score=scores[int(i)], rank=r, method='tfidf')
                    for r, i in enumerate(idx, start=1)]
        raise RuntimeError(f'No searchable sparse index found for {self.source}')

class DenseIndexAdapter:
    def __init__(self, index_path, meta_path, source, shared_docs=None, dim=384, space='cosine'):
        self.source = source
        self.index_path = Path(index_path)
        self.meta_path = Path(meta_path)
        meta = joblib.load(self.meta_path)
        meta_docs = extract_docs_from_loaded(meta)
        if shared_docs is not None and meta_docs is not None and len(shared_docs) == len(meta_docs):
            self.docs = shared_docs
            del meta_docs, meta
            gc.collect()
            print(f'[DenseIndexAdapter] {source}: reusing BM25 docs; dense meta docs released')
        else:
            self.docs = meta_docs
        if self.docs is None:
            raise ValueError(f'Could not extract dense docs from {meta_path}')
        self.index = hnswlib.Index(space=space, dim=dim)
        self.index.load_index(str(self.index_path))
        self.index.set_ef(128)
        print(f'[DenseIndexAdapter] {source}: docs={len(self.docs)} dim={dim} space={space} ef=128')

    def search(self, query, top_k=40):
        vec = embedding_model.encode([query], normalize_embeddings=True, convert_to_numpy=True).astype('float32')
        labels, distances = self.index.knn_query(vec, k=top_k)
        labels = labels.reshape(-1)
        distances = distances.reshape(-1)
        # cosine distance: lower is better. Convert to similarity-ish score.
        scores = 1.0 - distances
        return [make_result_item(self.source, int(i), self.docs[int(i)], score=s, rank=r, method='dense')
                for r, (i, s) in enumerate(zip(labels, scores), start=1)]


In [ ]:
simplewiki_sparse = SparseIndexAdapter(LOCAL_INDEX_FILES['simplewiki_bm25'], source='simplewiki')
mem_report('after SimpleWiki BM25')
kelm_sparse = SparseIndexAdapter(LOCAL_INDEX_FILES['kelm_bm25'], source='kelm')
mem_report('after KELM BM25')

simplewiki_dense = DenseIndexAdapter(
    LOCAL_INDEX_FILES['simplewiki_dense_index'],
    LOCAL_INDEX_FILES['simplewiki_dense_meta'],
    source='simplewiki',
    shared_docs=simplewiki_sparse.docs,
)
mem_report('after SimpleWiki dense')

kelm_dense = DenseIndexAdapter(
    LOCAL_INDEX_FILES['kelm_dense_index'],
    LOCAL_INDEX_FILES['kelm_dense_meta'],
    source='kelm',
    shared_docs=kelm_sparse.docs,
)
mem_report('after KELM dense')

reranker = CrossEncoder(RERANKER_MODEL_NAME, device='cpu')
mem_report('after CPU reranker')
print('Embedding device:', getattr(embedding_model, 'device', 'unknown'))
print('Reranker device:', reranker.model.device)


## 8. Hybrid retrieval, RRF, reranker


In [ ]:
def hybrid_retrieve(query, top_k_bm25=TOP_K_BM25, top_k_dense=TOP_K_DENSE):
    result_lists = []
    result_lists.append(simplewiki_sparse.search(query, top_k=top_k_bm25))
    result_lists.append(kelm_sparse.search(query, top_k=top_k_bm25))
    result_lists.append(simplewiki_dense.search(query, top_k=top_k_dense))
    result_lists.append(kelm_dense.search(query, top_k=top_k_dense))
    return result_lists

def rrf_fusion(result_lists, k=RRF_K, top_k=RRF_TOP_K):
    scores = defaultdict(float)
    docs = {}
    sources = defaultdict(list)
    for results in result_lists:
        for rank, item in enumerate(results, start=1):
            doc_id = item['doc_id']
            scores[doc_id] += 1.0 / (k + rank)
            if doc_id not in docs:
                docs[doc_id] = dict(item)
            sources[doc_id].append(item.get('method'))
    ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:top_k]
    fused = []
    for doc_id, score in ranked:
        item = dict(docs[doc_id])
        item['rrf_score'] = float(score)
        item['matched_methods'] = sorted(set(m for m in sources[doc_id] if m))
        fused.append(item)
    return fused

def rerank(query, docs, top_k=LLM_CONTEXT_K):
    if not docs:
        return []
    docs = docs[:RERANK_TOP_K]
    pairs = [(query, d['text'][:1200]) for d in docs]
    scores = reranker.predict(pairs)
    ranked = sorted(zip(docs, scores), key=lambda x: float(x[1]), reverse=True)
    out = []
    for doc, score in ranked[:top_k]:
        item = dict(doc)
        item['reranker_score'] = float(score)
        out.append(item)
    return out

def retrieve_and_rerank(query):
    result_lists = hybrid_retrieve(query)
    fused = rrf_fusion(result_lists)
    return rerank(query, fused)


In [ ]:
# Smoke test retrieval
query = 'Who was the first president of the United States?'
docs = retrieve_and_rerank(query)
for i, d in enumerate(docs[:5], start=1):
    print('='*80)
    print(i, d.get('source'), d.get('method'), d.get('matched_methods'), d.get('reranker_score'))
    print(d['text'][:500])


## 9. Prompting and answer parsing


In [ ]:
def option_id_from_text(text, valid_ids):
    text = normalize_text(text).strip()
    # Prefer exact first integer.
    m = re.search(r'\b\d+\b', text)
    if m:
        val = int(m.group(0))
        if val in valid_ids:
            return val
    # Fallback for A/B/C/D if options are 1..4.
    letter_map = {'A': 1, 'B': 2, 'C': 3, 'D': 4}
    m = re.search(r'\b([ABCD])\b', text.upper())
    if m and letter_map[m.group(1)] in valid_ids:
        return letter_map[m.group(1)]
    return None

def get_question_text(question):
    return getattr(question, 'text', None) or getattr(question, 'question_text', None) or str(question)

def get_options(question):
    return getattr(question, 'options')

def build_rag_prompt(question, docs, competition_name):
    qtext = get_question_text(question)
    options = '\n'.join(f'{int(opt.id)}. {opt.text}' for opt in get_options(question))
    context = '\n\n'.join(
        f'[DOC {i} | {doc.get("source", "unknown")} | score={doc.get("reranker_score", 0):.3f}]\n{doc["text"][:DOC_MAX_CHARS]}'
        for i, doc in enumerate(docs[:LLM_CONTEXT_K], start=1)
    )
    return f"""You are answering a multiple-choice quiz question.
Category: {competition_name}

Use ONLY the context below.
Do not choose an answer only because it shares words with the context.
Prefer the option explicitly supported by the context.
Return ONLY the numeric option id.

Question:
{qtext}

Options:
{options}

Context:
{context}

Answer:"""

def llm_choose_option(question, docs, competition_name):
    valid_ids = {int(opt.id) for opt in get_options(question)}
    prompt = build_rag_prompt(question, docs, competition_name)
    raw = run_local_llm(prompt, max_new_tokens=MAX_NEW_TOKENS_FINAL)
    option_id = option_id_from_text(raw, valid_ids)
    if option_id is None:
        option_id = int(get_options(question)[0].id)
        strategy = 'llm_invalid_output_fallback_first_option'
    else:
        strategy = 'hybrid_rag_rrf_rerank_qwen35_gguf'
    return option_id, {
        'strategy': strategy,
        'raw_llm_output': raw,
        'prompt_version': PROMPT_VERSION,
    }


## 10. Maths-only tools and LLM tool router


In [ ]:
try:
    from agentic_tools_rewritten import choose_with_agentic_tools, choose_with_structured_tool_call
    AGENTIC_TOOLS_AVAILABLE = True
    print('agentic_tools imported')
except Exception as e:
    AGENTIC_TOOLS_AVAILABLE = False
    print('agentic_tools not available:', repr(e))

def build_tool_router_prompt(question):
    qtext = get_question_text(question)
    options = '\n'.join(f'{int(opt.id)}. {opt.text}' for opt in get_options(question))
    return f"""You are a conservative mathematical tool router.

Choose exactly one tool:
- solve_equation
- evaluate_expression
- modular_day
- prime_digit_sum
- percentage_greater
- no_tool

Return ONLY valid JSON.
Do not solve manually.
Do not simplify manually.
Do not guess missing values.

JSON schema examples:
{{"tool": "evaluate_expression", "args": {{"expression": "5*8+4"}}}}
{{"tool": "no_tool", "args": {{}}}}

Question:
{qtext}

Options:
{options}

JSON:"""

def safe_json_loads(text):
    text = normalize_text(text).strip()
    try:
        return json.loads(text)
    except Exception:
        pass
    m = re.search(r'\{.*\}', text, flags=re.S)
    if m:
        try:
            return json.loads(m.group(0))
        except Exception:
            return None
    return None

def llm_tool_router(question):
    if not AGENTIC_TOOLS_AVAILABLE:
        return None
    prompt = build_tool_router_prompt(question)
    raw = run_local_llm(prompt, max_new_tokens=MAX_NEW_TOKENS_ROUTER, stop=['\n\n', '<|im_end|>'])
    call = safe_json_loads(raw)
    if not call or call.get('tool') == 'no_tool':
        return None
    try:
        return choose_with_structured_tool_call(question, call)
    except TypeError:
        try:
            return choose_with_structured_tool_call(question=question, tool_call=call)
        except Exception:
            return None
    except Exception:
        return None

def try_math_tools(question, use_llm_router=True):
    if not AGENTIC_TOOLS_AVAILABLE:
        return None
    try:
        decision = choose_with_agentic_tools(question, fallback=lambda q: None)
        if decision is not None:
            return decision
    except Exception as e:
        print('deterministic math tools failed:', repr(e))
    if use_llm_router:
        return llm_tool_router(question)
    return None


## 11. Final answer_strategy


In [ ]:
def answer_strategy(question, competition_name: str):
    valid_ids = {int(opt.id) for opt in get_options(question)}

    # 1. Maths-only tools
    if competition_name == MATH_COMPETITION_NAME:
        decision = try_math_tools(question, use_llm_router=True)
        if decision is not None:
            try:
                opt_id = int(decision.option_id)
                if opt_id in valid_ids:
                    return opt_id, {
                        'strategy': getattr(decision, 'strategy', 'math_tools'),
                        'confidence': float(getattr(decision, 'confidence', 1.0)),
                        'explanation': getattr(decision, 'explanation', None),
                        'retrieved_context': [],
                        'prompt_version': PROMPT_VERSION,
                    }
            except Exception:
                pass

    # 2. RAG fallback / normal non-math route
    docs = retrieve_and_rerank(get_question_text(question))
    option_id, meta = llm_choose_option(question, docs, competition_name)
    if option_id not in valid_ids:
        option_id = int(get_options(question)[0].id)
        meta['strategy'] = 'invalid_option_id_fallback_first_option'
    meta['retrieved_context'] = docs
    return option_id, meta


## 12. Dummy tests


In [ ]:
class DummyOption:
    def __init__(self, id, text):
        self.id = id
        self.text = text

class DummyQuestion:
    def __init__(self, text, options, qid=0, level=1):
        self.id = qid
        self.text = text
        self.options = options
        self.level = level

q = DummyQuestion(
    text='Who was the first president of the United States?',
    options=[
        DummyOption(1, 'Abraham Lincoln'),
        DummyOption(2, 'George Washington'),
        DummyOption(3, 'Thomas Jefferson'),
        DummyOption(4, 'John Adams'),
    ],
)

option_id, meta = answer_strategy(q, 'Ancient History and Politics')
print('Predicted:', option_id)
print('Strategy:', meta.get('strategy'))
print('Raw LLM:', meta.get('raw_llm_output'))
for d in meta.get('retrieved_context', [])[:3]:
    print('DOC:', d.get('source'), d.get('reranker_score'), d['text'][:250])


In [ ]:
q_math = DummyQuestion(
    text='What is the value of the expression 5*8+4?',
    options=[
        DummyOption(1, '40'),
        DummyOption(2, '42'),
        DummyOption(3, '44'),
        DummyOption(4, '48'),
    ],
)

option_id, meta = answer_strategy(q_math, 'Maths')
print('Predicted:', option_id)
print('Meta:', meta)


## 13. PoliMillionaire API loop skeleton


In [ ]:
# Fill these before running.
API_URL = 'http://131.175.15.22:51111/'

# Colab Secret names. Change these if your secrets use different names.
USERNAME_SECRET_NAME = 'USERNAME'
PASSWORD_SECRET_NAME = 'PASSWORD'

# Optional manual fallback. Leave as None when using Colab Secrets.
USERNAME = None
PASSWORD = None

# Number of full game attempts to run for each competition/category.
N_ATTEMPTS_PER_COMPETITION = 1

# Single cumulative log file. Every run appends rows instead of overwriting it.
RUN_LOG_PATH = LOG_DIR / 'run_qwen35_gguf_all_competitions.csv'


def _read_colab_secret(secret_name):
    if not secret_name:
        return None
    try:
        if 'userdata' in globals() and userdata is not None:
            return userdata.get(secret_name)
    except Exception as e:
        print(f'Could not read Colab secret {secret_name}:', repr(e))
    return None


def setup_client():
    from millionaire_client import MillionaireClient
    username = USERNAME or _read_colab_secret(USERNAME_SECRET_NAME)
    password = PASSWORD or _read_colab_secret(PASSWORD_SECRET_NAME)
    if username is None or password is None:
        raise ValueError(
            'Set USERNAME/PASSWORD manually or create Colab Secrets named '
            f'{USERNAME_SECRET_NAME!r} and {PASSWORD_SECRET_NAME!r}'
        )
    client = MillionaireClient(API_URL)
    client.login(username, password)
    return client


def get_competitions(client):
    competitions = client.competitions.list_all()
    for comp in competitions:
        print(comp.id, comp.name, getattr(comp, 'max_levels', None))
    return competitions


def get_competition_names(client):
    return {comp.id: comp.name for comp in get_competitions(client)}


def _serialize_retrieved_context(meta):
    return json.dumps([
        {
            'source': d.get('source'),
            'idx': d.get('idx'),
            'reranker_score': d.get('reranker_score'),
            'text': d.get('text', '')[:500],
        }
        for d in meta.get('retrieved_context', [])
    ], ensure_ascii=False)


def append_logs(df, output_csv=RUN_LOG_PATH):
    if df is None or df.empty:
        return
    output_csv = Path(output_csv)
    output_csv.parent.mkdir(parents=True, exist_ok=True)
    write_header = not output_csv.exists() or output_csv.stat().st_size == 0
    df.to_csv(output_csv, mode='a', header=write_header, index=False)


def run_competition(client, comp_id, competition_names, attempt_number=None, run_id=None):
    competition_name = competition_names[comp_id]
    logs = []
    session_started_at = time.strftime('%Y-%m-%d %H:%M:%S')

    try:
        game = client.game.start(competition_id=comp_id)
    except Exception as e:
        return pd.DataFrame([{
            'run_id': run_id,
            'attempt_number': attempt_number,
            'session_started_at': session_started_at,
            'session_id': None,
            'competition_id': comp_id,
            'competition_name': competition_name,
            'error_message': repr(e),
        }])

    while game.in_progress:
        question = game.current_question
        if question is None:
            break

        start = time.time()
        try:
            option_id, meta = answer_strategy(question, competition_name)
            latency = time.time() - start
            result = game.answer(option_id)
            logs.append({
                'run_id': run_id,
                'attempt_number': attempt_number,
                'session_started_at': session_started_at,
                'session_id': getattr(game, 'session_id', None),
                'competition_id': comp_id,
                'competition_name': competition_name,
                'question_id': getattr(question, 'id', None),
                'level': getattr(question, 'level', None),
                'question_text': get_question_text(question),
                'options_json': json.dumps([(int(o.id), o.text) for o in get_options(question)], ensure_ascii=False),
                'chosen_option_id': option_id,
                'correct': getattr(result, 'correct', None),
                'timed_out': getattr(result, 'timed_out', None),
                'game_over': getattr(result, 'game_over', None),
                'earned_amount': getattr(result, 'earned_amount', None),
                'latency_seconds': latency,
                'strategy': meta.get('strategy'),
                'confidence': meta.get('confidence'),
                'explanation': meta.get('explanation'),
                'raw_llm_output': meta.get('raw_llm_output'),
                'prompt_version': PROMPT_VERSION,
                'retrieved_context': _serialize_retrieved_context(meta),
                'error_message': None,
            })
        except Exception as e:
            logs.append({
                'run_id': run_id,
                'attempt_number': attempt_number,
                'session_started_at': session_started_at,
                'session_id': getattr(game, 'session_id', None),
                'competition_id': comp_id,
                'competition_name': competition_name,
                'question_id': getattr(question, 'id', None),
                'level': getattr(question, 'level', None),
                'question_text': get_question_text(question),
                'options_json': json.dumps([(int(o.id), o.text) for o in get_options(question)], ensure_ascii=False),
                'error_message': repr(e),
            })
            break

    return pd.DataFrame(logs)


def run_all_competitions(
    client=None,
    attempts_per_competition=N_ATTEMPTS_PER_COMPETITION,
    output_csv=RUN_LOG_PATH,
    competition_ids=None,
):
    if client is None:
        client = setup_client()

    competitions = get_competitions(client)
    if competition_ids is not None:
        selected_ids = set(int(x) for x in competition_ids)
        competitions = [comp for comp in competitions if int(comp.id) in selected_ids]
    competition_names = {comp.id: comp.name for comp in competitions}

    run_id = time.strftime('%Y%m%d_%H%M%S')
    all_logs = []

    for attempt_number in range(1, int(attempts_per_competition) + 1):
        for comp in competitions:
            print(f'Run {run_id} | attempt {attempt_number}/{attempts_per_competition} | {comp.id}: {comp.name}')
            df_logs = run_competition(
                client=client,
                comp_id=comp.id,
                competition_names=competition_names,
                attempt_number=attempt_number,
                run_id=run_id,
            )
            append_logs(df_logs, output_csv=output_csv)
            all_logs.append(df_logs)
            print(f'Appended {len(df_logs)} rows to {output_csv}')

    if all_logs:
        return pd.concat(all_logs, ignore_index=True)
    return pd.DataFrame()





## 14. Memory diagnostics


In [ ]:
def full_memory_diagnostics():
    import sys
    print('='*100)
    mem_report('diagnostics')
    print('\nGPU MEMORY - nvidia-smi')
    try:
        get_ipython().system('nvidia-smi')
    except Exception:
        pass
    print('\nBIG PYTHON OBJECTS BY sys.getsizeof')
    objs = []
    for name, obj in list(globals().items()):
        try:
            objs.append((name, type(obj).__name__, sys.getsizeof(obj)))
        except Exception:
            pass
    for name, typ, size in sorted(objs, key=lambda x: x[2], reverse=True)[:60]:
        print(f'{name:40s} {typ:35s} {size/1024**2:10.2f} MB')

full_memory_diagnostics()
